<a href="https://colab.research.google.com/github/mateisuceveanu/ml-predictia-categoriei-produsului-pe-baza-titlului/blob/main/notebook/final_train_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

###Task - Sarcina 3 – Predicția categoriei produsului pe baza titlului
##Suceveanu Matei

###1. Create function for feature engineering

In [ ]:
import pandas as pd
import re
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer, StandardScaler

# Funcția universală de extracție
def extract_smart_features(df):
    s = df.iloc[:, 0].astype(str).str.lower()
    features = pd.DataFrame(index=df.index)

    # Feature 1: Coduri specifice frigiderelor Bosch (KGV, KGN, KGN)
    features['is_fridge_code'] = s.apply(lambda x: 1 if re.search(r'\bkg[vna]\d+', x) else 0)

    # Feature 2: Side by Side / American Style (Smeg/Samsung)
    features['is_sbs'] = s.apply(lambda x: 1 if 'sbs' in x or 'side by side' in x else 0)

    # Feature 3: Litri (Unitate specifică frigiderelor)
    features['has_litres'] = s.apply(lambda x: 1 if re.search(r'\d+\s?(l|ltr|litre|litri)\b', x) else 0)

    # Feature 4: RPM (Unitate specifică MAȘINILOR DE SPĂLAT - ajută la excludere)
    features['has_rpm'] = s.apply(lambda x: 1 if re.search(r'\d{3,}\s?rpm', x) else 0)

    # Feature 5: KG (Capacitate spălare - ajută la excludere)
    features['has_kg_load'] = s.apply(lambda x: 1 if re.search(r'\b\d{1,2}\s?kg\b', x) else 0)

    return features

###2. Load data & OVERSAMPLING & CLEANING

In [ ]:
# Încărcare date
df = pd.read_csv(r"C:\Task final LA\ml-predictia-categoriei-produsului-pe-baza-titlului\data\IMLP4_TASK_03-products_cleaned.csv")

# --- STRATEGIA DE AUR: OVERSAMPLING & CLEANING ---
# Adăugăm manual rânduri care să "bată" ponderea mașinilor de spălat vase
gold_standard = pd.DataFrame([
    {"product_title": "bosch serie 4 kgv39vl31g fridge freezer", "category_label": "fridge freezer"},
    {"product_title": "bosch kgv39vl31g", "category_label": "fridge freezer"},
    {"product_title": "smeg sbs8004po fridge freezer", "category_label": "fridge freezer"},
    {"product_title": "smeg sbs8004po side by side", "category_label": "fridge freezer"}
] * 50) # Multiplicăm de 50 de ori ca modelul să "ia aminte"

df = pd.concat([df, gold_standard], ignore_index=True)

X = df[['product_title']]
y = df['category_label']

###3. Pipeline and pkl file

In [ ]:
# Pipeline
preprocessor = ColumnTransformer([
    ('text_tfidf', TfidfVectorizer(
        ngram_range=(1, 3),
        stop_words=['serie', 'series', '4', '6', '8'], # Eliminăm cuvintele care induc confuzie între categorii
        min_df=1
    ), 'product_title'),
    ('smart_features', Pipeline([
        ('extractor', FunctionTransformer(extract_smart_features, validate=False)),
        ('scaler', StandardScaler())
    ]), ['product_title'])
])

final_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LinearSVC(class_weight='balanced', C=0.5, max_iter=10000))
])

print("Antrenare model cu corecții pentru Bosch/Smeg...")
final_pipeline.fit(X, y)

#Save pkl file
joblib.dump(final_pipeline, r"C:\Task final LA\ml-predictia-categoriei-produsului-pe-baza-titlului\data\final_model_v4.pkl")
print("Model v4 salvat!")

###4. Concluzie
##In aceasta parte de cod am folosit modelul ales precedent, la care am adaugat feature engineering deoarece nu se identificau bine produslele din anumte categorii.

##A fost necesar adaugarea unor exemple noi de produse in CSV pentru a invata modelul mai bine.

## De asemenea, am specificat anumite elemente identificatoare pentru anumite categorii de produse